In [1]:
import gc

import os
import glob
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader

import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim.lr_scheduler import ReduceLROnPlateau

from pandas.core.common import flatten
import matplotlib.pyplot as plt
from collections import Counter

from tqdm import tqdm
import collections
import os.path as osp
import pandas as pd
import numpy as np
import collections
from collections import defaultdict
from sklearn.metrics import f1_score
from pandas.core.common import flatten
from scipy.special import softmax


In [2]:
class COCOGraph:
    def __init__(self):
        self.num_nodes = 17
        # COCO physical connections (pairs of joint indices)
        self.edges = [
            (0, 1), (0, 2), (1, 3), (2, 4),  # Head/Face
            (0, 5), (0, 6),                  # Nose to shoulders
            (5, 7), (7, 9),                  # Left arm
            (6, 8), (8, 10),                 # Right arm
            (5, 11), (6, 12),                # Torso
            (11, 13), (13, 15),              # Left leg
            (12, 14), (14, 16)               # Right leg
        ]
        self.A = self._get_adjacency_matrix()

    def _get_adjacency_matrix(self):
        # Initialize a 17x17 matrix of zeros
        A = np.zeros((self.num_nodes, self.num_nodes))
        
        # Add physical connections (undirected graph)
        for i, j in self.edges:
            A[i, j] = 1
            A[j, i] = 1
            
        # Add self-connections (Identity matrix) so a node attends to itself
        A = A + np.eye(self.num_nodes)
        
        # Normalize the matrix (simplified normalization)
        rowsum = A.sum(axis=1)
        D_inv = np.diag(rowsum ** -1)
        A_norm = np.dot(D_inv, A)
        
        return torch.tensor(A_norm, dtype=torch.float32)

class STGCNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, A, stride=1, residual=True):
        super(STGCNBlock, self).__init__()
        
        # We register the Adjacency matrix as a buffer so it moves to GPU with the model
        self.register_buffer('A', A)
        
        # 1. Spatial Graph Convolution (using 1x1 2D Conv as a math trick to multiply features)
        self.spatial_conv = nn.Conv2d(in_channels, out_channels, kernel_size=(1, 1))
        
        # 2. Temporal Convolution (standard 2D conv over the time dimension)
        # Kernel size 9 is common for temporal fields (looks at 9 frames at once)
        self.temporal_conv = nn.Conv2d(out_channels, out_channels, kernel_size=(9, 1), 
                                       padding=(4, 0), stride=(stride, 1))
        
        self.relu = nn.ReLU()
        
        # Residual connection setup
        if not residual:
            self.residual = lambda x: 0
        elif in_channels == out_channels and stride == 1:
            self.residual = lambda x: x
        else:
            self.residual = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=(stride, 1)),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        res = self.residual(x)
        
        # Spatial Graph Convolution step
        # x shape: (Batch, Channels, Time, Vertices)
        x = self.spatial_conv(x)
        
        # Multiply by the adjacency matrix A. 
        # We use einsum for clean tensor multiplication over the Vertices dimension.
        # n: batch, c: channel, t: time, v: vertices, w: connected vertices
        x = torch.einsum('nctv,vw->nctw', x, self.A)
        
        # Temporal Convolution step
        x = self.temporal_conv(x)
        
        return self.relu(x + res)

class CustomSTGCN(nn.Module):
    def __init__(self, num_classes, in_channels=3, num_frames=18):
        super(CustomSTGCN, self).__init__()
        
        graph = COCOGraph()
        A = graph.A
        
        # Data Normalization layer (highly recommended for raw coordinate data)
        self.data_bn = nn.BatchNorm1d(in_channels * graph.num_nodes)
        
        # Stacking the ST-GCN blocks
        self.st_gcn_networks = nn.Sequential(
            STGCNBlock(in_channels, 64, A, residual=False),
            STGCNBlock(64, 64, A),
            STGCNBlock(64, 128, A, stride=2), # Stride 2 halves the time dimension
            STGCNBlock(128, 128, A),
            STGCNBlock(128, 256, A, stride=2),
            STGCNBlock(256, 256, A)
        )
        
        # Global Average Pooling & Final Classifier
        self.fcn = nn.Conv2d(256, num_classes, kernel_size=1)

    def forward(self, x):
        # x shape expects: (Batch, Channels, Time, Vertices, Persons)
        # For simplicity, we assume 1 Person (squeeze it out)
        if x.dim() == 5:
            # Keep only first person if present
            x = x[..., 0]  # -> (N, C, T, V)
        elif x.dim() != 4:
            raise ValueError(f"Expected input dims 4 or 5, got shape {tuple(x.shape)}")
            
        N, C, T, V = x.size()
        
        # Normalize the input data
        x = x.permute(0, 3, 1, 2).contiguous()      # (N, V, C, T)
        x = x.view(N, V * C, T)                     # (N, V*C, T)
        x = self.data_bn(x)
        x = x.view(N, V, C, T).permute(0, 2, 3, 1).contiguous()  # (N, C, T, V)
        
        # Pass through ST-GCN blocks
        x = self.st_gcn_networks(x)
        
        # Global Average Pooling (average over Time and Vertices)
        x = F.avg_pool2d(x, x.size()[2:])
        
        # Final classification
        x = self.fcn(x)
        x = x.view(x.size(0), -1) # Flatten to (Batch, Num_Classes)
        
        return x

In [7]:
import numpy as np
import torch
import torch.nn.functional as F
from collections import deque
import statistics
import cv2
from ultralytics import YOLO

# --- CONFIG ---
WINDOW_SIZE = 18
NUM_KEYPOINTS = 17
PITCH_TYPES = ["CH - Changeup", "FF - Fastball", "SI - Sinker", "SL - Slider", "ST - Sweeper"]
NUM_CLASSES = len(PITCH_TYPES)
CONFIDENCE_THRESHOLD = 0.5
KEYPOINT_CONF_THRESHOLD = 0.20

# --- DEVICE ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# --- LOAD MODELS (Outside the loop!) ---
# 1. Load YOLO Pose model
yolo_model = YOLO("D:/GitHub/BaseballPitch/modules/pitcher_detection/runs/pose/best.pt")

# 2. Load ST-GCN model
stgcn_model = CustomSTGCN(num_classes=NUM_CLASSES, in_channels=3)
stgcn_model.load_state_dict(
    torch.load(
        "D:/GitHub/BaseballPitch/modules/pitcher_detection/best_pitch_stgcn.pth",
        map_location=device,
    )
)
stgcn_model = stgcn_model.to(device)
stgcn_model.eval()

# COCO edge list for pose drawing
coco_edges = COCOGraph().edges

# --- FIFO BUFFERS ---
frame_buffer = deque(maxlen=WINDOW_SIZE)
prediction_buffer = deque(maxlen=6)

# --- VIDEO SETUP ---
video_path = "D:/GitHub/BaseballPitch/trimmed_woo_pitch_videos/PitchType-CH_Zone-1_PlayID-5e1e243b-7471-37bc-9c5c-04a02d7207ab_Date-2025-09-08.mp4"
cap = cv2.VideoCapture(video_path)

frame_idx = 0
current_prediction = "Buffering..."
smoothed_prediction = "Buffering..."
latest_conf = 0.0

# --- CLASSIFICATION LOOP ---
with torch.no_grad():
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        display_frame = frame.copy()

        # 1. Get Skeleton from current frame using YOLO
        results = yolo_model(frame, verbose=False)

        # Extract the keypoints (x, y, confidence)
        if len(results[0].keypoints) > 0:
            xy = results[0].keypoints.xy[0].cpu().numpy()           # (17, 2)
            conf = results[0].keypoints.conf[0].cpu().numpy()       # (17,)
            conf_col = conf.reshape(-1, 1)                          # (17, 1)

            # skeleton: (17, 3) = [x, y, conf]
            skeleton = np.concatenate([xy, conf_col], axis=1).astype(np.float32)

            # Draw keypoints
            for i, (x, y) in enumerate(xy):
                if conf[i] >= KEYPOINT_CONF_THRESHOLD:
                    cv2.circle(display_frame, (int(x), int(y)), 4, (0, 255, 0), -1)

            # Draw skeleton edges
            for a, b in coco_edges:
                if conf[a] >= KEYPOINT_CONF_THRESHOLD and conf[b] >= KEYPOINT_CONF_THRESHOLD:
                    x1, y1 = xy[a]
                    x2, y2 = xy[b]
                    cv2.line(
                        display_frame,
                        (int(x1), int(y1)),
                        (int(x2), int(y2)),
                        (255, 180, 0),
                        2,
                    )
        else:
            # Fallback: If no person is detected, pad with zeros
            skeleton = np.zeros((NUM_KEYPOINTS, 3), dtype=np.float32)

        # Add skeleton to buffer as a tensor
        frame_buffer.append(torch.tensor(skeleton, dtype=torch.float32))

        # 2. Check if buffer is full enough for classification
        if len(frame_buffer) == WINDOW_SIZE:
            # 1. Stack the frames
            # Assuming each skeleton in the buffer is a tensor of shape (17, 3)
            # frame_tensor shape becomes: (Time=18, Vertices=17, Channels=3)
            frame_tensor = torch.stack(list(frame_buffer))

            # 2. Reshape to match the Training Dataset exactly
            # Permute to (Channels, Time, Vertices) -> (2, 0, 1) mapping
            stgcn_input = frame_tensor.permute(2, 0, 1)

            # Add the Persons dimension at the end -> (3, 18, 17, 1)
            stgcn_input = stgcn_input.unsqueeze(-1)

            # Add the Batch dimension at the front -> (1, 3, 18, 17, 1)
            stgcn_input = stgcn_input.unsqueeze(0)

            # Move to the same device as your model (cuda or cpu)
            model_device = next(stgcn_model.parameters()).device
            stgcn_input = stgcn_input.to(model_device)

            # 3. Run inference safely
            logits = stgcn_model(stgcn_input)
            probabilities = F.softmax(logits, dim=1)
            confidence, predicted_class = torch.max(probabilities, dim=1)
            latest_conf = confidence.item()

            # 4. Confidence thresholding
            if latest_conf >= CONFIDENCE_THRESHOLD:
                current_prediction = PITCH_TYPES[predicted_class.item()]
            else:
                current_prediction = "Uncertain"

            # Store the prediction in the buffer
            prediction_buffer.append(current_prediction)

            # Temporal Smoothing
            if len(prediction_buffer) == prediction_buffer.maxlen:
                smoothed_prediction = statistics.mode(prediction_buffer)
                print(f"Frame {frame_idx:03d} | Pitch: {smoothed_prediction} (Conf: {latest_conf:.2f})")
        else:
            # Buffering...
            pass

        # Overlay prediction text
        cv2.putText(
            display_frame,
            f"Frame: {frame_idx}",
            (20, 30),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (255, 255, 255),
            2,
        )
        cv2.putText(
            display_frame,
            f"Current: {current_prediction} ({latest_conf:.2f})",
            (20, 65),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 255),
            2,
        )
        cv2.putText(
            display_frame,
            f"Smoothed: {smoothed_prediction}",
            (20, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2,
        )

        # Show in real time
        cv2.imshow("Pitch Classification (press q to quit)", display_frame)
        key = cv2.waitKey(1) & 0xFF
        if key == ord('q') or key == 27:
            break

        frame_idx += 1
        
cap.release()
cv2.destroyAllWindows()

Using device: cuda
Frame 022 | Pitch: CH - Changeup (Conf: 1.00)
Frame 023 | Pitch: CH - Changeup (Conf: 1.00)
Frame 024 | Pitch: CH - Changeup (Conf: 1.00)
Frame 025 | Pitch: CH - Changeup (Conf: 1.00)
Frame 026 | Pitch: CH - Changeup (Conf: 1.00)
Frame 027 | Pitch: CH - Changeup (Conf: 1.00)
Frame 028 | Pitch: CH - Changeup (Conf: 1.00)
Frame 029 | Pitch: CH - Changeup (Conf: 1.00)
Frame 030 | Pitch: CH - Changeup (Conf: 1.00)
Frame 031 | Pitch: CH - Changeup (Conf: 1.00)
Frame 032 | Pitch: CH - Changeup (Conf: 1.00)
Frame 033 | Pitch: CH - Changeup (Conf: 1.00)
Frame 034 | Pitch: CH - Changeup (Conf: 1.00)
Frame 035 | Pitch: CH - Changeup (Conf: 1.00)
Frame 036 | Pitch: CH - Changeup (Conf: 1.00)
Frame 037 | Pitch: CH - Changeup (Conf: 1.00)
Frame 038 | Pitch: CH - Changeup (Conf: 1.00)
Frame 039 | Pitch: CH - Changeup (Conf: 1.00)
Frame 040 | Pitch: CH - Changeup (Conf: 1.00)
Frame 041 | Pitch: CH - Changeup (Conf: 1.00)
Frame 042 | Pitch: CH - Changeup (Conf: 1.00)
Frame 043 | Pit